# YOLOv2 Traffic Detection Experiment — IB Extended Essay

**RQ:** To what extent does changing the parameters of the YOLO (You Only Look
Once) algorithm affect the decisions made by Apollo Go robotaxis to safely
handle traffic scenarios?

This single notebook does everything: installs the libraries, sets up
YOLOv2, lets you upload a traffic photo, runs the detector with different
parameter settings, and saves a table + chart + labelled images you can use
in your essay.

**How to use this notebook:** run the cells from top to bottom, one at a
time (click a cell, press `Shift + Enter`). Read the comments in each code
cell — they explain exactly what that cell is doing and why. Some cells will
ask you to do something (like upload a photo) before you move to the next
one.


## Step 1 — Install the libraries we need

We only need to add OpenCV — Colab already has numpy and matplotlib installed, and reinstalling them with a specific version tends to make `pip` very slow, so we leave them alone.

In [ ]:
# opencv-python-headless is the version of OpenCV without extra GUI
# features we don't need (Colab can't pop up its own windows anyway).
# --quiet just hides the long wall of installation text.
!pip install --quiet opencv-python-headless


## Step 2 — Recreate YOLOv2's config files

YOLOv2 needs two small text files to work:
- `yolov2.cfg` — describes the *shape* of the neural network (how many layers, what size, etc.)
- `coco.names` — the list of the 80 object names YOLOv2 was trained to recognise (e.g. "person", "car", "bicycle")

Both are plain text and small, so instead of downloading them separately, this cell just writes them straight to disk for you.

In [ ]:
import os

# Make a folder called "model" to keep the YOLO files together
os.makedirs("model", exist_ok=True)
os.makedirs("sample_images", exist_ok=True)
os.makedirs("results", exist_ok=True)

# %%writefile would also work here, but since the text is generated by
# this script we just open()/write() it directly.
yolov2_cfg_text = '[net]\n# Testing\nbatch=1\nsubdivisions=1\n# Training\n# batch=64\n# subdivisions=8\nwidth=608\nheight=608\nchannels=3\nmomentum=0.9\ndecay=0.0005\nangle=0\nsaturation = 1.5\nexposure = 1.5\nhue=.1\n\nlearning_rate=0.001\nburn_in=1000\nmax_batches = 500200\npolicy=steps\nsteps=400000,450000\nscales=.1,.1\n\n[convolutional]\nbatch_normalize=1\nfilters=32\nsize=3\nstride=1\npad=1\nactivation=leaky\n\n[maxpool]\nsize=2\nstride=2\n\n[convolutional]\nbatch_normalize=1\nfilters=64\nsize=3\nstride=1\npad=1\nactivation=leaky\n\n[maxpool]\nsize=2\nstride=2\n\n[convolutional]\nbatch_normalize=1\nfilters=128\nsize=3\nstride=1\npad=1\nactivation=leaky\n\n[convolutional]\nbatch_normalize=1\nfilters=64\nsize=1\nstride=1\npad=1\nactivation=leaky\n\n[convolutional]\nbatch_normalize=1\nfilters=128\nsize=3\nstride=1\npad=1\nactivation=leaky\n\n[maxpool]\nsize=2\nstride=2\n\n[convolutional]\nbatch_normalize=1\nfilters=256\nsize=3\nstride=1\npad=1\nactivation=leaky\n\n[convolutional]\nbatch_normalize=1\nfilters=128\nsize=1\nstride=1\npad=1\nactivation=leaky\n\n[convolutional]\nbatch_normalize=1\nfilters=256\nsize=3\nstride=1\npad=1\nactivation=leaky\n\n[maxpool]\nsize=2\nstride=2\n\n[convolutional]\nbatch_normalize=1\nfilters=512\nsize=3\nstride=1\npad=1\nactivation=leaky\n\n[convolutional]\nbatch_normalize=1\nfilters=256\nsize=1\nstride=1\npad=1\nactivation=leaky\n\n[convolutional]\nbatch_normalize=1\nfilters=512\nsize=3\nstride=1\npad=1\nactivation=leaky\n\n[convolutional]\nbatch_normalize=1\nfilters=256\nsize=1\nstride=1\npad=1\nactivation=leaky\n\n[convolutional]\nbatch_normalize=1\nfilters=512\nsize=3\nstride=1\npad=1\nactivation=leaky\n\n[maxpool]\nsize=2\nstride=2\n\n[convolutional]\nbatch_normalize=1\nfilters=1024\nsize=3\nstride=1\npad=1\nactivation=leaky\n\n[convolutional]\nbatch_normalize=1\nfilters=512\nsize=1\nstride=1\npad=1\nactivation=leaky\n\n[convolutional]\nbatch_normalize=1\nfilters=1024\nsize=3\nstride=1\npad=1\nactivation=leaky\n\n[convolutional]\nbatch_normalize=1\nfilters=512\nsize=1\nstride=1\npad=1\nactivation=leaky\n\n[convolutional]\nbatch_normalize=1\nfilters=1024\nsize=3\nstride=1\npad=1\nactivation=leaky\n\n\n#######\n\n[convolutional]\nbatch_normalize=1\nsize=3\nstride=1\npad=1\nfilters=1024\nactivation=leaky\n\n[convolutional]\nbatch_normalize=1\nsize=3\nstride=1\npad=1\nfilters=1024\nactivation=leaky\n\n[route]\nlayers=-9\n\n[convolutional]\nbatch_normalize=1\nsize=1\nstride=1\npad=1\nfilters=64\nactivation=leaky\n\n[reorg]\nstride=2\n\n[route]\nlayers=-1,-4\n\n[convolutional]\nbatch_normalize=1\nsize=3\nstride=1\npad=1\nfilters=1024\nactivation=leaky\n\n[convolutional]\nsize=1\nstride=1\npad=1\nfilters=425\nactivation=linear\n\n\n[region]\nanchors =  0.57273, 0.677385, 1.87446, 2.06253, 3.33843, 5.47434, 7.88282, 3.52778, 9.77052, 9.16828\nbias_match=1\nclasses=80\ncoords=4\nnum=5\nsoftmax=1\njitter=.3\nrescore=1\n\nobject_scale=5\nnoobject_scale=1\nclass_scale=1\ncoord_scale=1\n\nabsolute=1\nthresh = .6\nrandom=1\n'
with open("model/yolov2.cfg", "w") as f:
    f.write(yolov2_cfg_text)

coco_names_text = 'person\nbicycle\ncar\nmotorbike\naeroplane\nbus\ntrain\ntruck\nboat\ntraffic light\nfire hydrant\nstop sign\nparking meter\nbench\nbird\ncat\ndog\nhorse\nsheep\ncow\nelephant\nbear\nzebra\ngiraffe\nbackpack\numbrella\nhandbag\ntie\nsuitcase\nfrisbee\nskis\nsnowboard\nsports ball\nkite\nbaseball bat\nbaseball glove\nskateboard\nsurfboard\ntennis racket\nbottle\nwine glass\ncup\nfork\nknife\nspoon\nbowl\nbanana\napple\nsandwich\norange\nbroccoli\ncarrot\nhot dog\npizza\ndonut\ncake\nchair\nsofa\npottedplant\nbed\ndiningtable\ntoilet\ntvmonitor\nlaptop\nmouse\nremote\nkeyboard\ncell phone\nmicrowave\noven\ntoaster\nsink\nrefrigerator\nbook\nclock\nvase\nscissors\nteddy bear\nhair drier\ntoothbrush\n'
with open("model/coco.names", "w") as f:
    f.write(coco_names_text)

print("Wrote model/yolov2.cfg and model/coco.names")


## Step 3 — Download the YOLOv2 weights file (~194 MB)

This is YOLOv2's trained "knowledge" — the actual numbers it learned during training. It's too big to type/embed like the files above, so we download it. This can take a couple of minutes depending on Colab's connection speed.

**If this cell fails** (the original host, pjreddie.com, is sometimes unreliable), tell me and we'll find a mirror to download from instead.

In [ ]:
import os

weights_path = "model/yolov2.weights"

if os.path.exists(weights_path) and os.path.getsize(weights_path) > 100_000_000:
    print("yolov2.weights already downloaded, skipping.")
else:
    !curl -L -o model/yolov2.weights https://pjreddie.com/media/files/yolov2.weights
    print("Download finished. File size:")
    !ls -lh model/yolov2.weights


## Step 4 — Upload your traffic photo

Running this cell pops up a real "choose file" button. Pick a traffic scene photo from your computer (see the note below for where to get one). This cell automatically saves it as `sample_images/traffic_scene_1.jpg` no matter what your original file was called, so we won't hit filename mismatches.

**Where to get a photo:** ideally a frame from a real Apollo Go ride-along video (screenshot it, and note the video title/URL/timestamp so you can cite it in your essay), or a photo from an open self-driving dataset (e.g. BDD100K), or a free-to-use stock/street photo (Wikimedia Commons, Pexels) with a mix of cars and pedestrians. Keep the source link for your bibliography.

In [ ]:
from google.colab import files
import shutil

print("Choose your traffic photo file:")
uploaded = files.upload()          # opens the upload dialog

# `uploaded` is a dictionary of {filename: file_bytes} for whatever you
# picked. We just take the first (and normally only) file you uploaded.
original_name = list(uploaded.keys())[0]

# Copy/rename it to the exact path the rest of this notebook expects.
shutil.move(original_name, "sample_images/traffic_scene_1.jpg")
print(f"Saved your photo as sample_images/traffic_scene_1.jpg (was '{original_name}')")


## Step 5 — The YOLOv2 detector code

This is the core of the whole experiment. Every function below is explained line-by-line in the comments — read through it once so you understand what YOLO is actually doing, since you'll need to explain this in your essay's methodology section.

In [ ]:
# ---------------------------------------------------------------
# IMPORT THE LIBRARIES WE NEED
# ---------------------------------------------------------------
import cv2          # OpenCV - loads/saves images AND can run a YOLO
                     # (Darknet) neural network for us, so we don't have
                     # to write the neural network maths ourselves.
import numpy as np  # numpy - lets us work with the arrays (lists of
                     # numbers) that OpenCV's YOLO output comes in.

# ---------------------------------------------------------------
# FILE LOCATIONS
# ---------------------------------------------------------------
CONFIG_PATH = "model/yolov2.cfg"        # the network's "blueprint"
WEIGHTS_PATH = "model/yolov2.weights"   # the network's trained "knowledge"
NAMES_PATH = "model/coco.names"         # list of the 80 object names YOLO knows


# ---------------------------------------------------------------
# HELPER FUNCTIONS - each one does ONE small job
# ---------------------------------------------------------------

def load_class_names(names_path):
    """Read coco.names and return it as a plain Python list of strings,
    e.g. ["person", "bicycle", "car", ...]. The line number in the file
    matches the class ID number that YOLO outputs."""
    with open(names_path, "r") as f:
        class_names = f.read().strip().split("\n")
    return class_names


def load_yolo_network(config_path, weights_path):
    """Load the YOLOv2 network into OpenCV using the cfg (blueprint)
    and weights (trained knowledge) files, and return it."""
    net = cv2.dnn.readNetFromDarknet(config_path, weights_path)
    return net


def get_output_layer_names(net):
    """YOLO's network has many layers, but we only want the numbers
    that come out of the FINAL detection layer(s). This function asks
    OpenCV which layer names those are."""
    all_layer_names = net.getLayerNames()
    output_indexes = net.getUnconnectedOutLayers()
    output_layer_names = [all_layer_names[i - 1] for i in output_indexes.flatten()]
    return output_layer_names


def detect_objects(image, net, confidence_threshold, nms_threshold,
                    input_width, input_height):
    """
    Runs YOLO on ONE image and returns a clean Python list of detections.
    Each detection looks like:
        {"class_id": 2, "confidence": 0.87, "box": [x, y, w, h]}

    This is the single most important function in the whole notebook -
    everything about the experiment comes from calling this function
    with different parameter values.
    """
    image_height, image_width = image.shape[:2]

    # Turn the image into a "blob": resize it to (input_width, input_height),
    # scale pixel values from 0-255 down to 0.0-1.0, and swap BGR->RGB
    # colour order (OpenCV loads images as BGR, YOLO expects RGB).
    blob = cv2.dnn.blobFromImage(
        image, 1 / 255.0, (input_width, input_height),
        swapRB=True, crop=False
    )

    # Feed the blob into the network and run it forward.
    net.setInput(blob)
    output_layer_names = get_output_layer_names(net)
    layer_outputs = net.forward(output_layer_names)

    boxes = []          # [x, y, w, h] for each kept box
    confidences = []    # confidence score for each kept box
    class_ids = []       # which class (car/person/etc) for each box

    for output in layer_outputs:
        for detection in output:
            # The first 5 numbers are box info; everything after that is
            # one confidence score per class (80 numbers, one per COCO class).
            class_scores = detection[5:]
            class_id = np.argmax(class_scores)      # class with the highest score
            confidence = class_scores[class_id]      # that highest score

            # This is exactly where CONFIDENCE_THRESHOLD gets used:
            if confidence > confidence_threshold:
                # Box positions come as FRACTIONS of the image size (0-1),
                # so multiply by the real image width/height for pixels.
                box_center_x = int(detection[0] * image_width)
                box_center_y = int(detection[1] * image_height)
                box_w = int(detection[2] * image_width)
                box_h = int(detection[3] * image_height)

                # OpenCV wants the TOP-LEFT corner, not the centre.
                x = int(box_center_x - box_w / 2)
                y = int(box_center_y - box_h / 2)

                boxes.append([x, y, box_w, box_h])
                confidences.append(float(confidence))
                class_ids.append(class_id)

    # Remove duplicate/overlapping boxes on the same object
    # (this is exactly where NMS_THRESHOLD gets used).
    kept_indexes = cv2.dnn.NMSBoxes(
        boxes, confidences, confidence_threshold, nms_threshold
    )

    detections = []
    if len(kept_indexes) > 0:
        for i in kept_indexes.flatten():
            detections.append({
                "class_id": class_ids[i],
                "confidence": confidences[i],
                "box": boxes[i],
            })
    return detections


def draw_boxes(image, detections, class_names):
    """Draws a rectangle + text label for every detection straight onto
    the image, and returns the labelled image."""
    image = image.copy()
    for det in detections:
        x, y, w, h = det["box"]
        label = class_names[det["class_id"]]
        confidence = det["confidence"]
        text = f"{label} {confidence * 100:.0f}%"
        cv2.rectangle(image, (x, y), (x + w, y + h), (0, 255, 0), 2)
        cv2.putText(image, text, (x, max(y - 10, 0)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    return image


print("Functions defined. Loading class names and the YOLOv2 network "
      "(this can take a few seconds)...")
class_names = load_class_names(NAMES_PATH)
net = load_yolo_network(CONFIG_PATH, WEIGHTS_PATH)
print(f"Ready. YOLOv2 knows {len(class_names)} object classes.")


## Step 6 — Sanity check: run YOLO once with default settings

Before running the full experiment, let's just check everything works on your photo with normal settings.

In [ ]:
from IPython.display import Image, display

IMAGE_PATH = "sample_images/traffic_scene_1.jpg"

# These are the parameters my RQ investigates - see README.md in the
# project for what each one means.
CONFIDENCE_THRESHOLD = 0.5
NMS_THRESHOLD = 0.4
INPUT_WIDTH = 416
INPUT_HEIGHT = 416

image = cv2.imread(IMAGE_PATH)
if image is None:
    raise FileNotFoundError(
        f"Could not open '{IMAGE_PATH}'. Re-run the Step 4 upload cell."
    )

detections = detect_objects(image, net, CONFIDENCE_THRESHOLD, NMS_THRESHOLD,
                             INPUT_WIDTH, INPUT_HEIGHT)

print(f"Found {len(detections)} object(s):")
for det in detections:
    name = class_names[det["class_id"]]
    print(f"  - {name}: {det['confidence'] * 100:.1f}% confident")

labelled = draw_boxes(image, detections, class_names)
cv2.imwrite("results/sanity_check.jpg", labelled)
display(Image("results/sanity_check.jpg"))


## Step 7 — The actual experiment: vary the confidence threshold

This is the part that produces the data for the essay. We run YOLO on the *same* photo several times, changing only `CONFIDENCE_THRESHOLD` each time (everything else — the image, `NMS_THRESHOLD`, the input size — is kept constant, which is what makes this a controlled experiment). For each run we record how many objects were detected in total, and how many were "safety-critical" (person, car, bicycle, traffic light, etc — the object types that would actually change a robotaxi's driving decision).

In [ ]:
import csv

SAFETY_CRITICAL_CLASSES = [
    "person", "bicycle", "car", "motorbike", "bus", "truck",
    "traffic light", "stop sign",
]

# The independent variable: every value here gets tested, one at a time.
CONFIDENCE_VALUES_TO_TEST = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

# Controlled variables - kept the same across every run:
FIXED_NMS_THRESHOLD = 0.4
FIXED_INPUT_WIDTH = 416
FIXED_INPUT_HEIGHT = 416

original_image = cv2.imread(IMAGE_PATH)
results = []

for confidence_value in CONFIDENCE_VALUES_TO_TEST:
    print(f"Testing CONFIDENCE_THRESHOLD = {confidence_value} ...")

    image_copy = original_image.copy()
    detections = detect_objects(
        image_copy, net,
        confidence_value, FIXED_NMS_THRESHOLD,
        FIXED_INPUT_WIDTH, FIXED_INPUT_HEIGHT,
    )

    safety_detections = [
        d for d in detections
        if class_names[d["class_id"]] in SAFETY_CRITICAL_CLASSES
    ]
    safety_list_text = ", ".join(
        f"{class_names[d['class_id']]} ({d['confidence'] * 100:.0f}%)"
        for d in safety_detections
    )

    results.append({
        "confidence_threshold": confidence_value,
        "total_objects_detected": len(detections),
        "safety_critical_objects_detected": len(safety_detections),
        "safety_critical_detail": safety_list_text,
    })

    # Save a labelled image for this threshold - useful for a
    # "before/after" comparison figure in the essay.
    labelled = draw_boxes(image_copy, detections, class_names)
    cv2.imwrite(f"results/detected_conf_{confidence_value}.jpg", labelled)

    print(f"  -> total objects: {len(detections)}, "
          f"safety-critical objects: {len(safety_detections)}")

# Save the results table as a CSV for the essay
with open("results/experiment_results.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(results[0].keys()))
    writer.writeheader()
    writer.writerows(results)

print("\nDone! Results saved to results/experiment_results.csv")


## Step 8 — View the results table and chart

In [ ]:
import pandas as pd

df = pd.read_csv("results/experiment_results.csv")
df


In [ ]:
import matplotlib.pyplot as plt

thresholds = df["confidence_threshold"]
totals = df["total_objects_detected"]
safety_totals = df["safety_critical_objects_detected"]

x_positions = range(len(thresholds))
bar_width = 0.35

plt.figure(figsize=(8, 5))
plt.bar([x - bar_width / 2 for x in x_positions], totals,
        width=bar_width, label="All objects detected")
plt.bar([x + bar_width / 2 for x in x_positions], safety_totals,
        width=bar_width, label="Safety-critical objects detected")

plt.xticks(list(x_positions), [str(t) for t in thresholds])
plt.xlabel("Confidence threshold")
plt.ylabel("Number of objects detected")
plt.title("Effect of YOLOv2 confidence threshold on detections")
plt.legend()
plt.tight_layout()
plt.savefig("results/confidence_vs_detections.png")
plt.show()


Take a look at a couple of the labelled images side by side to *see* objects appearing/disappearing as the threshold changes (pick two values from the table above, e.g. a low one and a high one):

In [ ]:
from IPython.display import Image, display

display(Image("results/detected_conf_0.1.jpg"))  # low threshold: more detections
display(Image("results/detected_conf_0.9.jpg"))  # high threshold: fewer, more confident detections


## Step 9 — Download everything for your essay

This zips up the whole `results/` folder so you can download it in one go.

In [ ]:
from google.colab import files

!zip -rq results.zip results
files.download("results.zip")


---
## Using these results in your essay

See the "4. Using these results in the essay" section of this project's
`README.md` for:
- an Independent/Dependent/Controlled variables table to put in your
  methodology section
- a 4-step structure for turning these numbers into an argument
  (state the pattern → connect it to a driving decision → extend to
  Apollo Go → state limitations)
- a fill-in-the-numbers example paragraph

**Quick summary of what each parameter means, in case you need it while writing:**
- **`CONFIDENCE_THRESHOLD`** — how sure YOLO must be before it reports a
  detection at all. Controls false negatives (missed hazards) vs false
  positives (phantom detections).
- **`NMS_THRESHOLD`** — how aggressively overlapping boxes on the same
  object get merged into one. Mostly about not double-counting the same
  object.
- **`INPUT_WIDTH` / `INPUT_HEIGHT`** — the resolution YOLO actually
  "looks" at. Bigger = better at spotting small/far-away objects, but
  slower — which matters for a moving car needing real-time answers.
